In [10]:
# Implementacja pliku NeuralNetwork.ipynb
# w PyTorch zamiast TensorFlow
import os
import cv2
import matplotlib.pyplot as plt

# Wczytanie danych
original_faces_path = "../../../Dane/Sample/Default"
original_faces = []

spoof_faces_paths = ["../../../Dane/Sample/Paper", "../../../Dane/Sample/Phone", "../../../Dane/Sample/Tablet"]
spoof_faces = []

# Wczytanie oryginalnych twarzy
files = [f for f in os.listdir(original_faces_path) if f.endswith(".jpg") or f.endswith(".png")] # Tylko pliki JPG i PNG
for f in files:
    image = cv2.imread(os.path.join(original_faces_path, f))
    original_faces.append(image)

# Wczytanie podstawionych twarzy
files = []
for path in spoof_faces_paths:
    files.extend([os.path.join(path, f) for f in os.listdir(path) if f.endswith(".jpg") or f.endswith(".png")])

for f in files:
    image = cv2.imread(f)
    spoof_faces.append(image)

# Wyświetlenie przykładowych twarzy
plt.imshow(cv2.cvtColor(original_faces[0], cv2.COLOR_BGR2RGB))
plt.show()

plt.imshow(cv2.cvtColor(spoof_faces[0], cv2.COLOR_BGR2RGB))
plt.show()

In [11]:
# Wstępne przetwarzanie danych
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder

# Zmiana rozmiaru obrazów wejściowych
# [PARAMETR]
face_size = (64, 64)

# Przeskalowanie obrazów przedstawiających autentyczne twarze
original_faces_resized = []
for face in original_faces:
    original_faces_resized.append(cv2.resize(face, face_size))

# Przeskalowanie obrazów przedstawiających podstawione twarze
spoof_faces_resized = []
for face in spoof_faces:
    spoof_faces_resized.append(cv2.resize(face, face_size))

# Wyświetlenie przykładowych twarzy
plt.imshow(cv2.cvtColor(original_faces_resized[0], cv2.COLOR_BGR2RGB))
plt.show()

plt.imshow(cv2.cvtColor(spoof_faces_resized[0], cv2.COLOR_BGR2RGB))
plt.show()

# Zamiana obrazów na macierze zawierające wartości z przedziału [0, 1] zamiast [0, 255]
original_faces_normalized = np.array(original_faces_resized) / 255.0
spoof_faces_normalized = np.array(spoof_faces_resized) / 255.0

# Utworzenie kategorii, do których sieć będzie przydzielać obrazy
labels = np.array(["original"] * len(original_faces_normalized) + ["spoof"] * len(spoof_faces_normalized))
le = LabelEncoder().fit(labels)
labels_encoded = le.transform(labels)
labels_encoded = torch.tensor(labels_encoded, dtype=torch.int64)

print(labels_encoded)

In [12]:
# Przerabianie obrazów
from torchvision import transforms

# Utworzenie generatora przerabiającego obrazy
# [PARAMETRY]
aug = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomRotation(20),  # Rotacja o losowy kąt z przedziału [-20, 20] stopni
    transforms.RandomResizedCrop(size=face_size, scale=(0.85, 1.0)),    # Przycięcie obrazu do zdefiniowanego rozmiaru
    transforms.RandomHorizontalFlip(),  # Losowe odbicie lustrzane
    transforms.RandomAffine(degrees=0, translate=(0.2, 0.2), shear=0.15),   # Losowe przesunięcie obrazu
    transforms.ToTensor()
])

In [13]:
# Utworzenie zbiorów danych
from sklearn.model_selection import train_test_split

# Połączenie zbiorów danych z odpowiednimi kategoriami
original_faces_tensor = torch.stack([aug(face) for face in original_faces_normalized])
spoof_faces_tensor = torch.stack([aug(face) for face in spoof_faces_normalized])
faces_all = np.concatenate((original_faces_tensor, spoof_faces_tensor))

# Podział danych na zbiór treningowy i testowy
(trainX, testX, trainY, testY) = train_test_split(faces_all, labels_encoded, test_size=0.25, random_state=61185)

# Konwersja danych na tensory
trainX = torch.tensor(trainX, dtype=torch.float32)
testX = torch.tensor(testX, dtype=torch.float32)
trainY = torch.tensor(trainY, dtype=torch.long)
testY = torch.tensor(testY, dtype=torch.long)

# Wyświetlenie części zbiorów
print(trainX)

In [14]:
# Utworzenie modelu sieci neuronowej
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

# Model sieci neuronowej
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # CONV => RELU => CONV => RELU => POOL
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding='same')
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 16, kernel_size=3, padding='same')
        self.bn2 = nn.BatchNorm2d(16)
        self.pool1 = nn.MaxPool2d(kernel_size=2)
        self.dropout1 = nn.Dropout(0.25)

        # CONV => RELU => CONV => RELU => POOL
        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding='same')
        self.bn3 = nn.BatchNorm2d(32)
        self.conv4 = nn.Conv2d(32, 32, kernel_size=3, padding='same')
        self.bn4 = nn.BatchNorm2d(32)
        self.pool2 = nn.MaxPool2d(kernel_size=2)
        self.dropout2 = nn.Dropout(0.25)

        # FC => RELU
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32 * 16 * 16, 64)
        self.bn5 = nn.BatchNorm1d(64)
        self.dropout3 = nn.Dropout(0.5)

        # Warstwa wyjściowa
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):
        # CONV => RELU => CONV => RELU => POOL
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = self.dropout1(x)
    
        # CONV => RELU => CONV => RELU => POOL
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = self.dropout2(x)
    
        # FC => RELU
        x = self.flatten(x)
        x = F.relu(self.bn5(self.fc1(x)))
        x = self.dropout3(x)
    
        # Klasyfikator Softmax
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

In [15]:
# Utworzenie modelu
model = Net()
criterion = nn.CrossEntropyLoss()

# Utworzenie optimizera
# [PARAMETRY]
INIT_LR = 1e-3
EPOCHS = 25
optimizer = Adam(model.parameters(), lr=INIT_LR, weight_decay=INIT_LR / EPOCHS)

In [16]:
# Trenowanie modelu
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter

# Przygotowanie danych
train_dataset = TensorDataset(trainX, trainY)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TensorDataset(testX, testY)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Utworzenie obiektu do zapisywania logów z trenowania modelu
writer = SummaryWriter()

# Pętla trenująca model
model.train()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for epoch in range(EPOCHS):
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")
    writer.add_scalar('training loss', running_loss / len(train_loader), epoch)

    # Validation loss
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    val_loss /= len(test_loader)
    val_accuracy = 100 * correct / total
    print(f"Validation Loss: {val_loss}, Accuracy: {val_accuracy}%")
    writer.add_scalar('validation loss', val_loss, epoch)
    writer.add_scalar('validation accuracy', val_accuracy, epoch)

writer.close()

In [17]:
# Wypisanie wytrenowanych wag modelu
print("Model's state_dict:")
for param_tensor in model.state_dict():
    print(param_tensor, "\t", model.state_dict()[param_tensor].size())

# Zapisanie diagramu modelu
from torchviz import make_dot
dummy_input = torch.randn(1, 3, 64, 64).to(device)
dot = make_dot(model(dummy_input), params=dict(model.named_parameters()))
dot.render('model_visualization', format='png')

In [18]:
# Zapisanie modelu
torch.save(model.state_dict(), "model.pth")